# Question 2 - Create the SQL Agent

The agent first inspects the database schema, then generates and executes SQL using the existing `taxi.db`. PydanticAI validates the final answer as a `SQLResult`.

```text
User question
    ↓
PydanticAI Agent
    ↓
get_schema()
    ↓
run_sql(query)
    ↓
DuckDB trips table
    ↓
SQLResult
```

Run this notebook from `week3_homework/` with the project dependencies and an `OPENAI_API_KEY` available in the environment. The agent uses `openai:gpt-4o-mini`.

In [1]:
from sql_agent import agent, SQLResult, sql_tools
from pydantic_ai.messages import ToolCallPart, ToolReturnPart

question = "What's the average trip distance for rides with 2 passengers?"

In [2]:
# Jupyter supports top-level await; the script uses asyncio.run() instead.
result = await agent.run(question)
assert isinstance(result.output, SQLResult)
display(result.output.model_dump())

                 pydantic-ai v2.63.0 • Python 3.13.15

      / \        agent: agent • model: openai:gpt-4o-mini • output: SQLResult • tools: 2
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


{'sql_query': 'SELECT AVG(trip_distance) AS average_trip_distance \nFROM trips \nWHERE passenger_count = 2;',
 'result_text': 'average_trip_distance\n3.7827640377879224',
 'row_count': 1}

In [3]:
print("sql_query:")
print(result.output.sql_query)
print("result_text:")
print(result.output.result_text)
print("row_count:")
print(result.output.row_count)

sql_query:
SELECT AVG(trip_distance) AS average_trip_distance 
FROM trips 
WHERE passenger_count = 2;
result_text:
average_trip_distance
3.7827640377879224
row_count:
1


In [4]:
tool_calls = [
    part.tool_name
    for message in result.all_messages()
    for part in message.parts
    if isinstance(part, ToolCallPart)
    and part.tool_name in {"get_schema", "run_sql"}
]
print("Tool-call order:", " -> ".join(tool_calls))
assert tool_calls[0] == "get_schema"
assert "run_sql" in tool_calls

# Verify that schema inspection completed before SQL was requested.
schema_returned = False
for message in result.all_messages():
    for part in message.parts:
        if isinstance(part, ToolReturnPart) and part.tool_name == "get_schema":
            schema_returned = True
        if isinstance(part, ToolCallPart) and part.tool_name == "run_sql":
            assert schema_returned
        if isinstance(part, ToolReturnPart) and part.tool_name == "run_sql":
            sql_tool_result = str(part.content)
            print("DuckDB tool result:")
            print(sql_tool_result)

assert result.output.row_count == 1
assert result.output.result_text == sql_tool_result

Tool-call order: get_schema -> run_sql
DuckDB tool result:
average_trip_distance
3.7827640377879224


The following independent SQL check runs directly against DuckDB to verify the agent’s answer. The tools return column headers and at most 50 data rows.

In [5]:
import math
from sql_tools import con, setup_database

assert setup_database() == 2964624
print(sql_tools.get_schema())
assert len(sql_tools.run_sql("SELECT * FROM trips LIMIT 75").splitlines()) == 51

verification_query = (
    "SELECT AVG(trip_distance) AS average_trip_distance "
    "FROM trips WHERE passenger_count = 2"
)
expected_average = con.execute(verification_query).fetchone()[0]
agent_average = float(sql_tool_result.splitlines()[1])
assert math.isclose(agent_average, expected_average, rel_tol=1e-12)
print("\nIndependent DuckDB average:", expected_average)
print("Agent average matches the independent query.")

Loaded 2964624 rows
VendorID: INTEGER
tpep_pickup_datetime: TIMESTAMP
tpep_dropoff_datetime: TIMESTAMP
passenger_count: BIGINT
trip_distance: DOUBLE
RatecodeID: BIGINT
store_and_fwd_flag: VARCHAR
PULocationID: INTEGER
DOLocationID: INTEGER
payment_type: BIGINT
fare_amount: DOUBLE
extra: DOUBLE
mta_tax: DOUBLE
tip_amount: DOUBLE
tolls_amount: DOUBLE
improvement_surcharge: DOUBLE
total_amount: DOUBLE
congestion_surcharge: DOUBLE
Airport_fee: DOUBLE

Independent DuckDB average: 3.7827640377879255
Agent average matches the independent query.


**Average trip distance for rides with 2 passengers: `3.7827640377879224`.**

This is the actual DuckDB value returned through the agent’s `run_sql` tool. The agent called `get_schema()` before `run_sql()`, and its validated `SQLResult` contains one result row. The independent SQL check above confirms the average (floating-point aggregation can differ in the last few digits).